In [79]:
import pandas as pd
from collections import Counter
from collections import Counter
import numpy as np
from ase import Atoms
from dscribe.descriptors import SOAP
import numpy as np

import pandas as pd
import numpy as np
import pymatgen
from pymatgen.core import Element
from mendeleev import element

import ast

In [80]:

df_ABC3 = pd.read_csv("../ABC3_filtered.csv", usecols= ['formula_pretty','elements', 'band_gap','energy_above_hull','formation_energy_per_atom','density',"structure.sites"])
df_A2BCX6 = pd.read_csv("../A2BCX6_filtered.csv", usecols= ['formula_pretty','elements', 'band_gap','energy_above_hull','formation_energy_per_atom','density',"structure.sites"])

pd.set_option('display.max_rows', 100)  # Display 10 rows
pd.set_option('display.max_columns', None)  # Display all columns
#Merge the two pervoskites data frames and treat them as one 
df = pd.concat([df_ABC3, df_A2BCX6], axis=0,ignore_index=True)
df.to_csv("df.csv", index=False) 
print(df)

                     elements formula_pretty   density  \
0           ['Ag', 'Br', 'S']         Ag3SBr  5.967646   
1            ['Ag', 'I', 'S']          Ag3SI  6.576090   
2            ['Ag', 'I', 'S']          Ag3SI  6.373934   
3           ['Ag', 'As', 'O']         AgAsO3  5.320520   
4           ['Ag', 'Cl', 'O']         AgClO3  4.138469   
...                       ...            ...       ...   
1419  ['As', 'Bi', 'Mg', 'O']      Mg2BiAsO6  5.206103   
1420   ['Bi', 'Mg', 'O', 'P']       Mg2BiPO6  5.243912   
1421   ['C', 'Cl', 'Mg', 'O']       Mg2CClO6  1.674336   
1422   ['Bi', 'Mg', 'O', 'V']       Mg2VBiO6  4.813911   
1423  ['Fe', 'Mg', 'O', 'Si']    MgFe(SiO3)2  3.468191   

      formation_energy_per_atom  energy_above_hull  band_gap  \
0                     -0.305716           0.098312    0.2964   
1                     -0.260466           0.066818    0.5478   
2                     -0.210889           0.116396    0.6243   
3                     -1.243946           0.025

In [81]:
missing_fusion_heats = {
    "As": 24.44,  # kJ/mol
    "O" : 0.44,
    "N" : 0.72,
    "C" : 117,
    "Ru" : 38.59,
    "Tb" : 10.15,
    "Gd" : 10.05,
    "Dy" : 11.06,
    "Eu" : 9.21,
    "Ho" : 17.0,
    "Er" : 19.9,
    "Lu" : 22,
    "Tm" : 16.84
}

from mendeleev import element

def get_fusion_heat(element_symbol):
    elmt = element(element_symbol)
    if elmt.fusion_heat is None and element_symbol in missing_fusion_heats:
        return missing_fusion_heats[element_symbol]
    elif elmt.fusion_heat is None and element_symbol not in missing_fusion_heats:
        print(elmt.name, "is missing fusion heat value")
    else:
        return elmt.fusion_heat

missing_evaporation_heats = {
    "O" : 6.82, # kJ/mol
    "N" : 5.58,
    "C" : 715,
    "Ru" : 619,
    "Tb" : 391
}

from mendeleev import element

def get_evaporation_heat(element_symbol):
    elmt = element(element_symbol)
    if elmt.evaporation_heat is None and element_symbol in missing_evaporation_heats:
        return missing_evaporation_heats[element_symbol]
    else:
        return elmt.evaporation_heat

missing_specific_heats = {
    "Pa" : 99.1,
    "Tc" : 63
}

def get_specific_heat(element_symbol):
    elmt = element(element_symbol)
    if elmt.specific_heat is None and element_symbol in missing_specific_heats:
        return missing_specific_heats[element_symbol]
    else:
        return elmt.specific_heat

def get_boiling_point(element_symbol):
    elmt = element(element_symbol)
    if element_symbol == "Pa":
        return 4000
    elif element_symbol == "P":
        return 550
    else:
        return elmt.boiling_point
    


def get_atomic_radius_calculated(elmt_pymatgen_temp):
    atomic_radius_calculated_temp = elmt_pymatgen_temp.atomic_radius_calculated
    if atomic_radius_calculated_temp == None:
        return elmt_pymatgen_temp.atomic_radius
    else :
      return atomic_radius_calculated_temp

def get_density_of_solid(elmt_pymatgen_temp):
    density_of_solid_temp = elmt_pymatgen_temp.density_of_solid
    if density_of_solid_temp == None:
        return elmt_pymatgen_temp.atomic_mass / elmt_pymatgen_temp.molar_volume *1000
    else :
      return density_of_solid_temp

In [82]:
#create dictionary with the values for all elements (so that the loop over all formulas is faster) (takes approx 4 minutes now)

#find all elements in the csv file to create input for dictionary creation
elements_list = pd.read_csv('df.csv', usecols= ['elements'])
elements_vector = []
for index in elements_list.index: # Iterate over DataFrame index to get row
  formula = elements_list.loc[index, 'elements'] # Access element string using loc
  formula = ast.literal_eval(formula) # Safely evaluate string as list
  for elmt in formula:
    elements_vector.append(elmt)

elements_vector = pd.Series(elements_vector).drop_duplicates().tolist() # Deduplicate elements


#loop over the found unique elements and define the needed properties
elemental_properties = {}  #dictionary to be filled

for symbol in elements_vector:
    #define strings as "elements"
    elmt_pymatgen = Element(symbol)
    elmt_mendeleev = element(symbol)

    properties = {
        'atomic_mass': elmt_pymatgen.atomic_mass,
        'Z': elmt_pymatgen.Z,
        'group': elmt_pymatgen.group,
        'fusion_heat': get_fusion_heat(symbol),  # Use custom function
        'evaporation_heat': get_evaporation_heat(symbol),  # Use custom function
        'specific_heat': get_specific_heat(symbol), # Use custom function
        'row': elmt_pymatgen.row,
        'mendeleev_no': elmt_pymatgen.mendeleev_no,
        'iupac_ordering': elmt_pymatgen.iupac_ordering,
        'average_ionic_radius': elmt_pymatgen.average_ionic_radius,
        'atomic_radius': elmt_pymatgen.atomic_radius,
        'atomic_radius_calculated': get_atomic_radius_calculated(elmt_pymatgen), # Use custom function
        'van_der_waals_radius': elmt_pymatgen.van_der_waals_radius,
        'X': elmt_pymatgen.X,
        'electron_affinity': elmt_pymatgen.electron_affinity,
        'ionization_energy': elmt_pymatgen.ionization_energy,
    #    'valence': elmt_pymatgen.valence[1],
        'density_of_solid': get_density_of_solid(elmt_pymatgen), # Use custom function
        'molar_volume': elmt_pymatgen.molar_volume,
        'boiling_point': get_boiling_point(symbol), # Use custom function
        'melting_point': elmt_pymatgen.melting_point,
        'thermal_conductivity': elmt_pymatgen.thermal_conductivity

    }

    elemental_properties[symbol] = properties  # For dictionary


/var/folders/k8/4c7hpqw11gq276z_k941vshc0000gn/T/ipykernel_90596/251560036.py:76: UserWarning: No data available for density_of_solid for Br
  density_of_solid_temp = elmt_pymatgen_temp.density_of_solid
/var/folders/k8/4c7hpqw11gq276z_k941vshc0000gn/T/ipykernel_90596/251560036.py:76: UserWarning: No data available for density_of_solid for O
  density_of_solid_temp = elmt_pymatgen_temp.density_of_solid
/var/folders/k8/4c7hpqw11gq276z_k941vshc0000gn/T/ipykernel_90596/251560036.py:76: UserWarning: No data available for density_of_solid for Cl
  density_of_solid_temp = elmt_pymatgen_temp.density_of_solid
/var/folders/k8/4c7hpqw11gq276z_k941vshc0000gn/T/ipykernel_90596/251560036.py:76: UserWarning: No data available for density_of_solid for N
  density_of_solid_temp = elmt_pymatgen_temp.density_of_solid
/var/folders/k8/4c7hpqw11gq276z_k941vshc0000gn/T/ipykernel_90596/251560036.py:76: UserWarning: No data available for density_of_solid for F
  density_of_solid_temp = elmt_pymatgen_temp.densi

In [83]:
df = pd.read_csv('df.csv', usecols= ['formula_pretty','elements', 'band_gap','energy_above_hull','formation_energy_per_atom','density','structure.sites'])

#loop over all entries and extract the corresponding elements
index = -1
for formula in df["elements"]:
  index += 1
  #reformat to list
  elements = ast.literal_eval(formula)

  #calculate property and mean/sd
  atomic_mass = []
  Z = []
  group = []
  row = []
  mendeleev_no = []
  iupac_ordering = []
  average_ionic_radius = []
  atomic_radius = []
  atomic_radius_calculated = []
  van_der_waals_radius = []
  X = []
  electron_affinity = []
  ionization_energy = []
  valence = []
  atomic_mass = []
  density_of_solid = []
  molar_volume = []
  fusion_heat = []
  evaporation_heat = []
  specific_heat = []

#use the library to assign the properties
  for elmt in elements:

    atomic_mass.append(elemental_properties[elmt]['atomic_mass'])
    Z.append(elemental_properties[elmt]['Z'])
    group.append(elemental_properties[elmt]['group'])
    row.append(elemental_properties[elmt]['row'])
    mendeleev_no.append(elemental_properties[elmt]['mendeleev_no'])
    iupac_ordering.append(elemental_properties[elmt]['iupac_ordering'])
    average_ionic_radius.append(elemental_properties[elmt]['average_ionic_radius'])
    atomic_radius.append(elemental_properties[elmt]['atomic_radius'])
    atomic_radius_calculated.append(elemental_properties[elmt]['atomic_radius_calculated'])
    van_der_waals_radius.append(elemental_properties[elmt]['van_der_waals_radius'])
    X.append(elemental_properties[elmt]['X'])
    electron_affinity.append(elemental_properties[elmt]['electron_affinity'])
    ionization_energy.append(elemental_properties[elmt]['ionization_energy'])
    #valence.append(elmt.valence[1])
    density_of_solid.append(elemental_properties[elmt]['density_of_solid'])
    molar_volume.append(elemental_properties[elmt]['molar_volume'])
    fusion_heat.append(elemental_properties[elmt]['fusion_heat'])
    evaporation_heat.append(elemental_properties[elmt]['evaporation_heat'])
    specific_heat.append(elemental_properties[elmt]['specific_heat'])

  mean_atomic_mass = np.mean(atomic_mass)
  std_atomic_mass = np.std(atomic_mass)
  mean_Z = np.mean(Z)
  std_Z = np.std(Z)
  mean_group = np.mean(group)
  std_group = np.std(group)
  mean_row = np.mean(row)
  std_row = np.std(row)
  mean_mendeleev_no = np.mean(mendeleev_no)
  std_mendeleev_no = np.std(mendeleev_no)
  mean_iupac_ordering = np.mean(iupac_ordering)
df = pd.read_csv('df.csv', usecols= ['formula_pretty','elements', 'band_gap','energy_above_hull','formation_energy_per_atom','density','structure.sites'])

#loop over all entries and extract the corresponding elements
index = -1
for formula in df["elements"]:
  index += 1
  #reformat to list
  elements = ast.literal_eval(formula)

  #calculate property and mean/sd
  atomic_mass = []
  Z = []
  group = []
  row = []
  mendeleev_no = []
  iupac_ordering = []
  average_ionic_radius = []
  atomic_radius = []
  atomic_radius_calculated = []
  van_der_waals_radius = []
  X = []
  electron_affinity = []
  ionization_energy = []
  valence = []
  atomic_mass = []
  density_of_solid = []
  molar_volume = []
  fusion_heat = []
  evaporation_heat = []
  specific_heat = []

#use the library to assign the properties
  for elmt in elements:

    atomic_mass.append(elemental_properties[elmt]['atomic_mass'])
    Z.append(elemental_properties[elmt]['Z'])
    group.append(elemental_properties[elmt]['group'])
    row.append(elemental_properties[elmt]['row'])
    mendeleev_no.append(elemental_properties[elmt]['mendeleev_no'])
    iupac_ordering.append(elemental_properties[elmt]['iupac_ordering'])
    average_ionic_radius.append(elemental_properties[elmt]['average_ionic_radius'])
    atomic_radius.append(elemental_properties[elmt]['atomic_radius'])
    atomic_radius_calculated.append(elemental_properties[elmt]['atomic_radius_calculated'])
    van_der_waals_radius.append(elemental_properties[elmt]['van_der_waals_radius'])
    X.append(elemental_properties[elmt]['X'])
    electron_affinity.append(elemental_properties[elmt]['electron_affinity'])
    ionization_energy.append(elemental_properties[elmt]['ionization_energy'])
    #valence.append(elmt.valence[1])
    density_of_solid.append(elemental_properties[elmt]['density_of_solid'])
    molar_volume.append(elemental_properties[elmt]['molar_volume'])
    fusion_heat.append(elemental_properties[elmt]['fusion_heat'])
    evaporation_heat.append(elemental_properties[elmt]['evaporation_heat'])
    specific_heat.append(elemental_properties[elmt]['specific_heat'])

  mean_atomic_mass = np.mean(atomic_mass)
  std_atomic_mass = np.std(atomic_mass)
  mean_Z = np.mean(Z)
  std_Z = np.std(Z)
  mean_group = np.mean(group)
  std_group = np.std(group)
  mean_row = np.mean(row)
  std_row = np.std(row)
  mean_mendeleev_no = np.mean(mendeleev_no)
  std_mendeleev_no = np.std(mendeleev_no)
  mean_iupac_ordering = np.mean(iupac_ordering)
  std_iupac_ordering = np.std(iupac_ordering)
df = pd.read_csv('df.csv', usecols= ['formula_pretty','elements', 'band_gap','energy_above_hull','formation_energy_per_atom','density','structure.sites'])

#loop over all entries and extract the corresponding elements
index = -1
for formula in df["elements"]:
  index += 1
  #reformat to list
  elements = ast.literal_eval(formula)

  #calculate property and mean/sd
  atomic_mass = []
  Z = []
  group = []
  row = []
  mendeleev_no = []
  iupac_ordering = []
  average_ionic_radius = []
  atomic_radius = []
  atomic_radius_calculated = []
  van_der_waals_radius = []
  X = []
  electron_affinity = []
  ionization_energy = []
  valence = []
  atomic_mass = []
  density_of_solid = []
  molar_volume = []
  fusion_heat = []
  evaporation_heat = []
  specific_heat = []

#use the library to assign the properties
  for elmt in elements:

    atomic_mass.append(elemental_properties[elmt]['atomic_mass'])
    Z.append(elemental_properties[elmt]['Z'])
    group.append(elemental_properties[elmt]['group'])
    row.append(elemental_properties[elmt]['row'])
    mendeleev_no.append(elemental_properties[elmt]['mendeleev_no'])
    iupac_ordering.append(elemental_properties[elmt]['iupac_ordering'])
    average_ionic_radius.append(elemental_properties[elmt]['average_ionic_radius'])
    atomic_radius.append(elemental_properties[elmt]['atomic_radius'])
    atomic_radius_calculated.append(elemental_properties[elmt]['atomic_radius_calculated'])
    van_der_waals_radius.append(elemental_properties[elmt]['van_der_waals_radius'])
    X.append(elemental_properties[elmt]['X'])
    electron_affinity.append(elemental_properties[elmt]['electron_affinity'])
    ionization_energy.append(elemental_properties[elmt]['ionization_energy'])
    #valence.append(elmt.valence[1])
    density_of_solid.append(elemental_properties[elmt]['density_of_solid'])
    molar_volume.append(elemental_properties[elmt]['molar_volume'])
    fusion_heat.append(elemental_properties[elmt]['fusion_heat'])
    evaporation_heat.append(elemental_properties[elmt]['evaporation_heat'])
    specific_heat.append(elemental_properties[elmt]['specific_heat'])

  mean_atomic_mass = np.mean(atomic_mass)
  std_atomic_mass = np.std(atomic_mass)
  mean_Z = np.mean(Z)
  std_Z = np.std(Z)
  mean_group = np.mean(group)
  std_group = np.std(group)
  mean_row = np.mean(row)
  std_row = np.std(row)
  mean_mendeleev_no = np.mean(mendeleev_no)
  std_mendeleev_no = np.std(mendeleev_no)
  mean_iupac_ordering = np.mean(iupac_ordering)
  std_iupac_ordering = np.std(iupac_ordering)
  mean_average_ionic_radius = np.mean(average_ionic_radius)
  std_average_ionic_radius = np.std(average_ionic_radius)
  mean_atomic_radius = np.mean(atomic_radius)
  std_atomic_radius = np.std(atomic_radius)
  mean_atomic_radius_calculated = np.mean(atomic_radius_calculated)
  std_atomic_radius_calculated = np.std(atomic_radius_calculated)
  mean_van_der_waals_radius = np.mean(van_der_waals_radius)
  std_van_der_waals_radius = np.std(van_der_waals_radius)
  mean_X = np.mean(X)
  std_X = np.std(X)
  mean_electron_affinity = np.mean(electron_affinity)
  std_electron_affinity = np.std(electron_affinity)
  mean_ionization_energy = np.mean(ionization_energy)
  std_ionization_energy = np.std(ionization_energy)
  #mean_valence = np.mean(valence)   ##valence not working yet because of ambiguity
  #std_valence = np.std(valence)
  mean_density_of_solid = np.mean(density_of_solid)
  std_density_of_solid = np.std(density_of_solid)
  mean_molar_volume = np.mean(molar_volume)
  std_molar_volume = np.std(molar_volume)
  mean_fusion_heat = np.mean(fusion_heat)
  std_fusion_heat = np.std(fusion_heat)
  mean_evaporation_heat = np.mean(evaporation_heat)
  std_evaporation_heat = np.std(evaporation_heat)
  mean_specific_heat = np.mean(specific_heat)
  std_specific_heat = np.std(specific_heat)

  #add to dataframe
  df.at[index, 'atom_mass_mean'] = mean_atomic_mass
  df.at[index, 'atom_mass_std'] = std_atomic_mass
  df.at[index, 'Z_mean'] = mean_Z
  df.at[index, 'Z_std'] = std_Z
  df.at[index, 'group_mean'] = mean_group
  df.at[index, 'group_std'] = std_group
  df.at[index, 'row_mean'] = mean_row
  df.at[index, 'row_std'] = std_row
  df.at[index, 'mendeleev_no_mean'] = mean_mendeleev_no
  df.at[index, 'mendeleev_no_std'] = std_mendeleev_no
  df.at[index, 'iupac_ordering_mean'] = mean_iupac_ordering
  df.at[index, 'iupac_ordering_std'] = std_iupac_ordering
  df.at[index, 'average_ionic_radius_mean'] = mean_average_ionic_radius
  df.at[index, 'average_ionic_radius_std'] = std_average_ionic_radius
  df.at[index, 'atomic_radius_mean'] = mean_atomic_radius
  df.at[index, 'atomic_radius_std'] = std_atomic_radius
  df.at[index, 'atomic_radius_calculated_mean'] = mean_atomic_radius_calculated
  df.at[index, 'atomic_radius_calculated_std'] = std_atomic_radius_calculated
  df.at[index, 'van_der_waals_radius_mean'] = mean_van_der_waals_radius
  df.at[index, 'van_der_waals_radius_std'] = std_van_der_waals_radius
  df.at[index, 'X_mean'] = mean_X
  df.at[index, 'X_std'] = std_X
  df.at[index, 'electron_affinity_mean'] = mean_electron_affinity
  df.at[index, 'electron_affinity_std'] = std_electron_affinity
  df.at[index, 'ionization_energy_mean'] = mean_ionization_energy
  df.at[index, 'ionization_energy_std'] = std_ionization_energy
  #df.at[index, 'valence_mean'] = mean_valence
  #df.at[index, 'valence_std'] = std_valence
  df.at[index, 'density_of_solid_mean'] = mean_density_of_solid
  df.at[index, 'density_of_solid_std'] = std_density_of_solid
  df.at[index, 'molar_volume_mean'] = mean_molar_volume
  df.at[index, 'molar_volume_std'] = std_molar_volume
  df.at[index, 'fusion_heat_mean'] = mean_fusion_heat
  df.at[index, 'fusion_heat_std'] = std_fusion_heat
  df.at[index, 'evaporation_heat_mean'] = mean_evaporation_heat
  df.at[index, 'evaporation_heat_std'] = std_evaporation_heat
  df.at[index, 'specific_heat_mean'] = mean_specific_heat
  df.at[index, 'specific_heat_std'] = std_specific_heat
# Display the updated dataframe
#df

In [84]:

def get_data (df):
    # expects a dataframe with the following columns: "elements", "structure.sites"

    # returns a dataframe where the elements and structure.sites string are plit and unnecessary character removed

    df_curated = pd.DataFrame()
    df_curated["Elements"] = df["elements"]
    # df_curated["band_gap"] = df["band_gap"]
    # df_curated["formula_pretty"] = df["formula_pretty"]
    df_curated["structure_string"] = df["structure.sites"]
    df_curated["structure_string_split"] = None

    for i in range(df_curated.shape[0]):
        

        df_curated["structure_string_split"].iloc[i] = df_curated["structure_string"].iloc[i].split()
        # #cleans the structure string and removes all the unnecessary characters
        for j in range(len(df_curated["structure_string_split"].iloc[i])):


            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace("{","")
            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace("}","")
            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace("[","")
            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace("]","")
            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace(":","")
            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace("'","")
            df_curated["structure_string_split"].iloc[i][j] = df_curated["structure_string_split"].iloc[i][j].replace(",","")

        df_curated["Elements"].iloc[i] = df_curated["Elements"].iloc[i].split(",")
        for k in range(len(df_curated["Elements"].iloc[i])):
            df_curated["Elements"].iloc[i][k] = df_curated["Elements"].iloc[i][k].replace("'", "")
            df_curated["Elements"].iloc[i][k] = df_curated["Elements"].iloc[i][k].replace("[", "")
            df_curated["Elements"].iloc[i][k] = df_curated["Elements"].iloc[i][k].replace("]", "")
            df_curated["Elements"].iloc[i][k] = df_curated["Elements"].iloc[i][k].replace(" ", "")


    return df_curated


def split_df (df):
    # expects a dataframe with the following columns: "structure_string_split" where each elements has 18 entires
    # select only the positons with relevant data and store them as a touple of (element, [x,y,z])
    # returns the data frame with the appended column "positions" with the positions of the atoms in the structure


    df["positions"] = None
    for i in range(len(df["structure_string_split"])):
        position_i = []
        for j in range (0, len(df["structure_string_split"].iloc[i]), 18):
            element = df["structure_string_split"].iloc[i][j+2] # remove trailing comma
            x = float(df["structure_string_split"].iloc[i][j+15])
            y = float(df["structure_string_split"].iloc[i][j+16])
            z = float(df["structure_string_split"].iloc[i][j+17])
            position_i.append((element, [x, y, z]))
        df["positions"].iloc[i] = position_i
    return df


def get_ratio(df):
    # Input: pd.datafram with a column "positions" containing lists of tuples (element, [x, y, z])
    #
    # Output: appends dataframe with a new column "ratio" containing a sorted dictionary of elements and their ratios
    #
    df["ratio"] = None

    #loops over all entriey in positions
    for i, atom_list in enumerate(df["positions"]):
        elements = [atom[0] for atom in atom_list]
        occurance = Counter(elements)

        counts = np.fromiter(occurance.values(), dtype=int)

        gcd = np.gcd.reduce(counts)


        reduced_occurance = {element: count // gcd for element, count in occurance.items()}

        # Sort by value so largest is last
        sorted_occurrence = dict(sorted(reduced_occurance.items(), key=lambda item: item[1]))

        df["ratio"].iloc[i] = sorted_occurrence

    return df


def get_ase(df):
    # expect a data frame with a column "positions" containing lists of tuples (element, [x, y, z])
    # returns a data frame with a new column "ase" containing ase atoms objects




    df["ase"] = None  # Optional, for clarity
    for i, row in df.iterrows():
        symbols = [atom[0] for atom in row["positions"]]
        positions = [atom[1] for atom in row["positions"]]
        
        atoms_obj = Atoms(symbols=symbols, positions=positions)
        df.at[i, "ase"] = atoms_obj
    return df


def get_soap(df, r_cut=190.0, n_max=2, l_max=2, sigma=1):
    # Automatically extract all unique species from df["Elements"]


    unique_species = sorted(set(e for row in df["Elements"] for e in row))

    # Create SOAP descriptor generator
    soap_generator = SOAP(
        species=unique_species,
        r_cut=r_cut,
        n_max=n_max,
        l_max=l_max,
        sigma=sigma,
        periodic=False,
        sparse=False,
        rbf = "gto"
    )

    # Generate and store SOAP descriptors
    soap_list = []
    for atoms in df["ase"]:
        descriptor = soap_generator.create(atoms)
        #this seems to make every soap have the same length
        avg_descriptor = np.mean(descriptor, axis=0) #check if this is the correct way to do it with stehpan 

        soap_list.append(avg_descriptor)

    df["soap"] = soap_list
    return df


def append_soap(df):
    # expects a dataframe with the following columns: "elements", "structure.sites"
 
    # retrun the datafram with the soap descriptor column appended at the end 
    df_soap = df.copy()
    df_soap = get_data(df_soap)
    df_soap = split_df(df_soap)
    df_soap = get_ase(df_soap)
    df_soap = get_soap(df_soap)
    df["elements"] = df_soap["Elements"]
    df_soap.drop(columns=["Elements","structure_string", "structure_string_split","positions","ase"], inplace=True)
    df = pd.concat([df, df_soap], axis=1)
    df.drop(columns=["structure.sites"], inplace=True)
    return df


df_soap = append_soap(df)
df_soap



/var/folders/k8/4c7hpqw11gq276z_k941vshc0000gn/T/ipykernel_90596/3893413360.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_curated["structure_string_split"].iloc[i] = df_curated["structure_string"].iloc[i].split()
/var/folders/k8/4c

,elements,formula_pretty,density,formation_energy_per_atom,energy_above_hull,band_gap,atom_mass_mean,atom_mass_std,Z_mean,Z_std,group_mean,group_std,row_mean,row_std,mendeleev_no_mean,mendeleev_no_std,iupac_ordering_mean,iupac_ordering_std,average_ionic_radius_mean,average_ionic_radius_std,atomic_radius_mean,atomic_radius_std,atomic_radius_calculated_mean,atomic_radius_calculated_std,van_der_waals_radius_mean,van_der_waals_radius_std,X_mean,X_std,electron_affinity_mean,electron_affinity_std,ionization_energy_mean,ionization_energy_std,density_of_solid_mean,density_of_solid_std,molar_volume_mean,molar_volume_std,fusion_heat_mean,fusion_heat_std,evaporation_heat_mean,evaporation_heat_std,specific_heat_mean,specific_heat_std,soap
0,"[Ag, Br, S]",Ag3SBr,5.967646,-0.305716,0.098312,0.2964,73.279067,31.299080,32.666667,12.762793,14.666667,2.624669,4.000000,0.816497,87.666667,11.897712,89.333333,12.364825,0.949722,0.096840,1.250000,0.254951,1.156667,0.349698,1.920000,0.135892,2.490000,0.425284,2.248389,0.849311,9.916685,1.758155,5496.545332,3631.543621,15.193333,3.889733,7.916667,4.761634,98.053333,110.615679,0.472333,0.193105,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"[Ag, I, S]",Ag3SI,6.576090,-0.260466,0.066818,0.5478,88.945890,40.964795,38.666667,16.213849,14.666667,2.624669,4.333333,0.942809,87.333333,11.614168,89.000000,12.083046,1.080000,0.160647,1.333333,0.249444,1.226667,0.318991,1.963333,0.127105,2.390000,0.326905,2.146875,0.717999,9.462501,1.334313,5796.666667,3534.650698,17.173333,6.413581,9.566667,6.072409,102.183333,108.185892,0.385667,0.228085,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"[Ag, I, S]",Ag3SI,6.373934,-0.210889,0.116396,0.6243,88.945890,40.964795,38.666667,16.213849,14.666667,2.624669,4.333333,0.942809,87.333333,11.614168,89.000000,12.083046,1.080000,0.160647,1.333333,0.249444,1.226667,0.318991,1.963333,0.127105,2.390000,0.326905,2.146875,0.717999,9.462501,1.334313,5796.666667,3534.650698,17.173333,6.413581,9.566667,6.072409,102.183333,108.185892,0.385667,0.228085,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,"[Ag, As, O]",AgAsO3,5.320520,-1.243946,0.025765,0.5930,66.263067,38.001726,29.333333,16.131405,14.000000,2.160247,3.666667,1.247219,87.000000,12.328828,86.000000,10.424331,1.002222,0.252122,1.116667,0.408928,1.090000,0.478957,1.826667,0.241431,2.516667,0.660824,1.190024,0.280016,10.327613,2.495842,5712.874808,3906.285741,13.526667,2.923062,12.276667,9.800681,97.773333,111.031840,0.494000,0.302259,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"[Ag, Cl, O]",AgClO3,4.138469,-0.407245,0.038285,2.3983,53.106867,39.528162,24.000000,16.673332,14.666667,2.624669,3.333333,1.247219,90.333333,13.695092,90.000000,12.832251,1.042222,0.198463,1.066667,0.410961,0.973333,0.494930,1.793333,0.242808,2.843333,0.655862,2.126104,1.053143,11.387307,2.707886,4483.441609,4271.691505,15.006667,3.349352,6.266667,4.700031,93.776667,113.501396,0.544000,0.282596,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1419,"[As, Bi, Mg, O]",Mg2BiAsO6,5.206103,-2.305771,0.006046,3.3424,81.051600,77.225838,34.000000,29.841247,12.000000,5.787918,3.750000,1.479020,87.500000,9.937303,72.250000,32.690786,0.953750,0.221059,1.212500,0.391112,1.125000,0.392078,1.792500,0.199045,2.237500,0.767606,0.696990,0.689803,9.584589,2.517608,4541.656106,3528.824303,16.405000,3.266960,11.270000,8.589173,85.755000,68.256561,0.598000,0.381432,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1420,"[Bi, Mg, O, P]",Mg2BiPO6,5.243912,-2.709739,0.000785,3.6043,70.064641,80.378293,29.500000,30.987901,12.000000,5.787918,3.500000,1.500000,87.750000,9.984363,72.500000,32.821487,0.926250,0.259407,1.175000,0.402337,1.085000,0.396642,1.780000,0.196596,2.240000,0.767431,0.682521,0.688001,9.759123,2.549692,3565.656106,3605.071106,17.422500,2.597695,5.787500

In [85]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from matplotlib import pyplot as plt
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, RandomizedSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score, RocCurveDisplay
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
%matplotlib inline 
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn import ensemble
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV 


In [ ]:
X = df_soap.select_dtypes(include=[np.number]).drop(columns=["band_gap"])

y = np.asarray(df_soap["band_gap"])

print(X.shape, y.shape)
print(X)



X



(1424, 39) (1424,)
       density  formation_energy_per_atom  energy_above_hull  atom_mass_mean  \
0     5.967646                  -0.305716           0.098312       73.279067   
1     6.576090                  -0.260466           0.066818       88.945890   
2     6.373934                  -0.210889           0.116396       88.945890   
3     5.320520                  -1.243946           0.025765       66.263067   
4     4.138469                  -0.407245           0.038285       53.106867   
...        ...                        ...                ...             ...   
1419  5.206103                  -2.305771           0.006046       81.051600   
1420  5.243912                  -2.709739           0.000785       70.064641   
1421  1.674336                  -1.461314           0.506400       21.942025   
1422  4.813911                  -2.523292           0.007843       75.056575   
1423  3.468191                  -2.909883           0.002547       31.058725   

      atom_mass_std 

,density,formation_energy_per_atom,energy_above_hull,atom_mass_mean,atom_mass_std,Z_mean,Z_std,group_mean,group_std,row_mean,row_std,mendeleev_no_mean,mendeleev_no_std,iupac_ordering_mean,iupac_ordering_std,average_ionic_radius_mean,average_ionic_radius_std,atomic_radius_mean,atomic_radius_std,atomic_radius_calculated_mean,atomic_radius_calculated_std,van_der_waals_radius_mean,van_der_waals_radius_std,X_mean,X_std,electron_affinity_mean,electron_affinity_std,ionization_energy_mean,ionization_energy_std,density_of_solid_mean,density_of_solid_std,molar_volume_mean,molar_volume_std,fusion_heat_mean,fusion_heat_std,evaporation_heat_mean,evaporation_heat_std,specific_heat_mean,specific_heat_std
0,5.967646,-0.305716,0.098312,73.279067,31.299080,32.666667,12.762793,14.666667,2.624669,4.000000,0.816497,87.666667,11.897712,89.333333,12.364825,0.949722,0.096840,1.250000,0.254951,1.156667,0.349698,1.920000,0.135892,2.490000,0.425284,2.248389,0.849311,9.916685,1.758155,5496.545332,3631.543621,15.193333,3.889733,7.916667,4.761634,98.053333,110.615679,0.472333,0.193105
1,6.576090,-0.260466,0.066818,88.945890,40.964795,38.666667,16.213849,14.666667,2.624669,4.333333,0.942809,87.333333,11.614168,89.000000,12.083046,1.080000,0.160647,1.333333,0.249444,1.226667,0.318991,1.963333,0.127105,2.390000,0.326905,2.146875,0.717999,9.462501,1.334313,5796.666667,3534.650698,17.173333,6.413581,9.566667,6.072409,102.183333,108.185892,0.385667,0.228085
2,6.373934,-0.210889,0.116396,88.945890,40.964795,38.666667,16.213849,14.666667,2.624669,4.333333,0.942809,87.333333,11.614168,89.000000,12.083046,1.080000,0.160647,1.333333,0.249444,1.226667,0.318991,1.963333,0.127105,2.390000,0.326905,2.146875,0.717999,9.462501,1.334313,5796.666667,3534.650698,17.173333,6.413581,9.566667,6.072409,102.183333,108.185892,0.385667,0.228085
3,5.320520,-1.243946,0.025765,66.263067,38.001726,29.333333,16.131405,14.000000,2.160247,3.666667,1.247219,87.000000,12.328828,86.000000,10.424331,1.002222,0.252122,1.116667,0.408928,1.090000,0.478957,1.826667,0.241431,2.516667,0.660824,1.190024,0.280016,10.327613,2.495842,5712.874808,3906.285741,13.526667,2.923062,12.276667,9.800681,97.773333,111.031840,0.494000,0.302259
4,4.138469,-0.407245,0.038285,53.106867,39.528162,24.000000,16.673332,14.666667,2.624669,3.333333,1.247219,90.333333,13.695092,90.000000,12.832251,1.042222,0.198463,1.066667,0.410961,0.973333,0.494930,1.793333,0.242808,2.843333,0.655862,2.126104,1.053143,11.387307,2.707886,4483.441609,4271.691505,15.006667,3.349352,6.266667,4.700031,93.776667,113.501396,0.544000,0.282596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1419,5.206103,-2.305771,0.006046,81.051600,77.225838,34.000000,29.841247,12.000000,5.787918,3.750000,1.479020,87.500000,9.937303,72.250000,32.690786,0.953750,0.221059,1.212500,0.391112,1.125000,0.392078,1.792500,0.199045,2.237500,0.767606,0.696990,0.689803,9.584589,2.517608,4541.656106,3528.824303,16.405000,3.266960,11.270000,8.589173,85.755000,68.256561,0.598000,0.381432
1420,5.243912,-2.709739,0.000785,70.064641,80.378293,29.500000,30.987901,12.000000,5.787918,3.500000,1.500000,87.750000,9.984363,72.500000,32.821487,0.926250,0.259407,1.175000,0.402337,1.085000,0.396642,1.780000,0.196596,2.240000,0.767431,0.682521,0.688001,9.759123,2.549692,3565.656106,3605.071106,17.422500,2.597695,5.787500,4.420211,90.105000,65.203812,0.708000,0.350158
1421,1.674336,-1.461314,0.506400,21.942025,8.973239,10.750000,4.205651,12.250000,6.015605,2.500000,0.500000,92.000000,11.180340,75.000000,34.503623,0.800000,0.341174,0.950000,0.350000,0.847500,0.364991,1.675000,0.091241,2.615000,0.819283,1.478990,1.432336,11.373053,2.317574,1741.331207,509.088997,13.510000,4.941796,33.262500,48.449319,218.507500,290.722037,0.782250,0.208396
1422,4.813911,-2.523292,0.007843,75.056575,78.391007,31.500000,30.236567,9.500000,6.103278,3.750000,1.479020,78.750000,17.383541,63.750000,31.633

In [110]:
from sklearn.ensemble import RandomForestRegressor
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.1, random_state=9)

for col in X_train.columns:
    if X_train[col].apply(lambda x: isinstance(x, (list, dict, tuple, set))).any():
        print(f"Non-scalar values found in column: {col}")

In [111]:

rfr = GridSearchCV (RandomForestRegressor(),{'max_depth':[None], 'random_state':[0],}, cv=5)
rfr.fit(X_train, y_train)
rfr_score = rfr.score(X_train,y_train)
rfr_score1 = rfr.score(X_test,y_test)
y_predicted8 = rfr.predict(X_test)
print('RFR Model| R2 sq on train set: %.4f'% rfr_score)
print('RFR Model| R2 sq on test set: %.4f'% rfr_score1)
print('RFR Model| MSE on test set: %.4f'% mean_squared_error(y_test, y_predicted8))
print('RFR Model| MAE on test set: %.4f'% mean_absolute_error(y_test, y_predicted8))

RFR Model| R2 sq on train set: 0.9594
RFR Model| R2 sq on test set: 0.7289
RFR Model| MSE on test set: 0.6408
RFR Model| MAE on test set: 0.6078


In [ ]:

# Parameter grid for hyperparameter search

p_grid = {"n_estimators": [10,100,1000], "max_depth": [None,10,20,30]}
# Initialise the Random Forest classifier
rf = RandomForestRegressor(random_state=22)

# Initialise KFold
kf = KFold(n_splits=5, shuffle=True, random_state=12)

# GridSearch CV
gs = GridSearchCV(estimator=rf, param_grid=p_grid, cv=kf, scoring='neg_mean_squared_error')

gs.fit(X_train, y_train)
RMSE_neg = cross_val_score(gs, X=X_train, y=y_train, cv=kf, scoring='neg_mean_squared_error')
RMSE = np.sqrt(-RMSE_neg)

# Define model with best hyperparameters and print them
model = gs.best_estimator_
print(f"Hyperparameters: {gs.best_params_}\n")


# p_grid = {"n_estimators": [10,15,30], "max_depth": [None,10,20]}
# Hyperparameters: {'max_depth': 20, 'n_estimators': 30}


In [131]:
from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate_model_with_cv(model, X, y, random_state):
    """
    Evaluate a model using 5-fold cross-validation and return performance metrics.

    Parameters:
        model: The model to evaluate
        X: Feature matrix
        y: Target vector
        random_state: Random seed for reproducibility

    Returns:
        Dictionary with performance metrics and arrays of per-fold metrics
    """
    # Initialize KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

    # Lists to store metrics
    metrics = {
        'RMSEs': [],
    }

    # Create figure for ROC curves
    plt.figure()

    # Iterate over folds
    for k, (train_index, test_index) in enumerate(kf.split(X)):
        # Split data
        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]

        # Train model
        model.fit(X_train_fold, y_train_fold)

        # Predict
        y_pred_fold = model.predict(X_test_fold)

        # Calculate metrics
        metrics['RMSEs'].append(mean_squared_error(y_test_fold, y_pred_fold))
     

        # ROC curve

    # Print metrics
    metric_display_names = {
        'RMSEs': 'RMSE',
     
    }
    for metric_name, values in metrics.items():
        display_name = metric_display_names[metric_name]
        print(f"Average {display_name}: {np.mean(values):.4f} \t and std err: {stats.sem(values):.4f}")

    return metrics

In [132]:
metrics_rf = evaluate_model_with_cv(model, X, y, random_state=22)

Average RMSE: 1.2821 	 and std err: 0.0581


<Figure size 640x480 with 0 Axes>

In [133]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=24)

In [136]:
rf = RandomForestRegressor(n_estimators=30, max_depth=None, random_state=22)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
print("MAE:", mean_absolute_error(y_val, y_pred))
print("R2:", r2_score(y_val, y_pred))
print("MSE:", mean_squared_error(y_val, y_pred))
#RMSE: 1.0720059864583598
# MAE: 0.8087310000000001
# R2: 0.529266025551792
# MSE: 1.149196835002561

MAE: 0.8239588940809969
R2: 0.5068127082611729
MSE: 1.2040118315107997
